# 0. data 구조 파악

In [1]:
# 0. Google Drive mount
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import zipfile, os, pandas as pd

zip_path = '/content/drive/MyDrive/26sp_ML/PR1/GTSRB.zip'
extract_dir = '/content/GTSRB'

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print(os.listdir(extract_dir))

train_df = pd.read_csv(os.path.join(extract_dir, 'Train.csv'))
test_df  = pd.read_csv(os.path.join(extract_dir, 'Test.csv'))
meta_df  = pd.read_csv(os.path.join(extract_dir, 'Meta.csv'))

print('Train columns:', train_df.columns.tolist())
print('Test columns :', test_df.columns.tolist())
print('Meta columns :', meta_df.columns.tolist())

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
print('Meta shape :', meta_df.shape)

print(train_df.head())
print(test_df.head())

print('num classes:', train_df['ClassId'].nunique())
print('train class range:', sorted(train_df['ClassId'].unique())[:5], '...', sorted(train_df['ClassId'].unique())[-5:])
print('sample train paths:', train_df['Path'].head().tolist())
print('sample test paths :', test_df['Path'].head().tolist())

Mounted at /content/drive
['Train', 'Meta.csv', 'meta', 'train', 'Test.csv', 'Test', 'Train.csv', 'test', 'Meta']
Train columns: ['Width', 'Height', 'Roi.X1', 'Roi.Y1', 'Roi.X2', 'Roi.Y2', 'ClassId', 'Path']
Test columns : ['Width', 'Height', 'Roi.X1', 'Roi.Y1', 'Roi.X2', 'Roi.Y2', 'ClassId', 'Path']
Meta columns : ['Path', 'ClassId', 'ShapeId', 'ColorId', 'SignId']
Train shape: (39209, 8)
Test shape : (12630, 8)
Meta shape : (43, 5)
   Width  Height  Roi.X1  Roi.Y1  Roi.X2  Roi.Y2  ClassId  \
0     27      26       5       5      22      20       20   
1     28      27       5       6      23      22       20   
2     29      26       6       5      24      21       20   
3     28      27       5       6      23      22       20   
4     28      26       5       5      23      21       20   

                             Path  
0  Train/20/00020_00000_00000.png  
1  Train/20/00020_00000_00001.png  
2  Train/20/00020_00000_00002.png  
3  Train/20/00020_00000_00003.png  
4  Train/20/000

In [2]:
# 1. Imports
import os
import zipfile
import copy
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.model_selection import train_test_split

In [3]:
# 2. Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# 3. Paths
zip_path = '/content/drive/MyDrive/26sp_ML/PR1/GTSRB.zip'
extract_dir = '/content/GTSRB'
save_dir = '/content/drive/MyDrive/26sp_ML/PR1'
os.makedirs(extract_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)

In [4]:
train_csv_path = os.path.join(extract_dir, 'Train.csv')
test_csv_path  = os.path.join(extract_dir, 'Test.csv')

train_df_full = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

train_df, val_df = train_test_split(
    train_df_full,
    test_size=0.1,
    random_state=42,
    stratify=train_df_full['ClassId']
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),                                              # Data augmentation
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),       # Data augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class GTSRBDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row['Path'])
        label = int(row['ClassId'])

        image = Image.open(img_path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [5]:
train_dataset = GTSRBDataset(train_df, extract_dir, transform=train_transform)
val_dataset = GTSRBDataset(val_df, extract_dir, transform=eval_transform)
test_dataset = GTSRBDataset(test_df, extract_dir, transform=eval_transform)

batch_size = 64
num_workers = 2
pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory
)

print('Train dataset size:', len(train_dataset))
print('Validation dataset size:', len(val_dataset))
print('Test dataset size:', len(test_dataset))

Train dataset size: 35288
Validation dataset size: 3921
Test dataset size: 12630


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

num_classes = train_df['ClassId'].nunique()

weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

def run_one_epoch(model, loader, criterion, optimizer=None, device='cpu'):
    if optimizer is None:
        model.eval()
    else:
        model.train()

    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if optimizer is not None:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            preds = outputs.argmax(dim=1)
            loss.backward()
            optimizer.step()

        else:
            with torch.no_grad():
                outputs = model(images)
                loss = criterion(outputs, labels)
                preds = outputs.argmax(dim=1)

        running_loss += loss.item() * images.size(0)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = running_correct / total
    return epoch_loss, epoch_acc

Using device: cuda


In [7]:
num_epochs = 10

best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

best_model_path = os.path.join(save_dir, 'resnet18_gtsrb_best_full_model.pth')

for epoch in range(num_epochs):
    train_loss, train_acc = run_one_epoch(model, train_loader, criterion, optimizer=optimizer, device=device)
    val_loss, val_acc = run_one_epoch(model, val_loader, criterion, optimizer=None, device=device)

    print(f"[Epoch {epoch+1:02d}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    # Best model 저장
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

        # 현재 best weight를 model에 넣은 뒤 전체 모델 저장
        model.load_state_dict(best_model_wts)
        model_cpu = copy.deepcopy(model).cpu()
        torch.save(model_cpu, best_model_path)
        model = model.to(device)

        print(f"Best model saved at epoch {epoch+1} -> {best_model_path}")

[Epoch 01/10] Train Loss: 0.2947, Train Acc: 0.9327 | Val Loss: 0.0138, Val Acc: 0.9974
Best model saved at epoch 1 -> /content/drive/MyDrive/26sp_ML/PR1/resnet18_gtsrb_best_full_model.pth
[Epoch 02/10] Train Loss: 0.0098, Train Acc: 0.9984 | Val Loss: 0.0121, Val Acc: 0.9969
[Epoch 03/10] Train Loss: 0.0051, Train Acc: 0.9992 | Val Loss: 0.0191, Val Acc: 0.9957
[Epoch 04/10] Train Loss: 0.0091, Train Acc: 0.9981 | Val Loss: 0.0044, Val Acc: 0.9992
Best model saved at epoch 4 -> /content/drive/MyDrive/26sp_ML/PR1/resnet18_gtsrb_best_full_model.pth
[Epoch 05/10] Train Loss: 0.0030, Train Acc: 0.9996 | Val Loss: 0.0028, Val Acc: 0.9995
Best model saved at epoch 5 -> /content/drive/MyDrive/26sp_ML/PR1/resnet18_gtsrb_best_full_model.pth
[Epoch 06/10] Train Loss: 0.0050, Train Acc: 0.9990 | Val Loss: 0.0078, Val Acc: 0.9980
[Epoch 07/10] Train Loss: 0.0082, Train Acc: 0.9982 | Val Loss: 0.0081, Val Acc: 0.9982
[Epoch 08/10] Train Loss: 0.0036, Train Acc: 0.9993 | Val Loss: 0.0013, Val Acc: 